# Week 09 期中評量：指定 chunk 產生器 (Assigned Chunks for DATA_REPORT 3.4)

本 notebook 會依照 **你的學號** 與 **老師在課堂上公布的 SEED**，從你自己的清洗結果中，指定兩個 chunk 給你分析：

1. **一個「很長」的 chunk**：從最長的 20 個 chunk 中選出一個。
2. **一個「資訊熵很低」的 chunk**：從資訊熵最低的 20 個 chunk 中選出一個。

每位同學拿到的 chunk 都不同。請把結果寫進 `DATA_REPORT.md` 第 3.4 節。

**驗證碼是什麼？** 程式把「學號 | SEED | chunk 類型」組成一段文字，計算 SHA-256 雜湊值 (`digest`)。
* 用雜湊值決定分配給你哪一個 chunk。
* **驗證碼就是 `digest[:8]`，也就是雜湊值的前 8 個字元。**
* 同樣的學號與 SEED，每次都得到同樣的驗證碼；換一個學號或 SEED，驗證碼就完全不同。老師會用你的學號與 SEED 重新計算來核對。

> **使用前提：** 你已在第 8 週完成 `01_Data_Ingestion.ipynb`，且電腦上的 `data/processed/` 中有 `cleaned_*_processed.json` 檔案。
> 本 notebook **不會**重新執行 Docling，只讀取已存好的 JSON 檔案，幾秒鐘就能跑完。

**使用方式：**
1. 把本檔案放在 `notebooks/` 資料夾 (與 `01_Data_Ingestion.ipynb` 同一個資料夾)，或專案根目錄。
2. 在下方填入你的學號與老師公布的 SEED。
3. 點選上方 **Run All (全部執行)**。

In [ ]:
# =========================================================
# 請填入 (Fill in)
# =========================================================
STUDENT_ID = ""   # 你的學號，例如 "M11412345"
SEED = ""         # 老師在第 9 週課堂上公布的 SEED

In [ ]:
import json, math, hashlib, os, re
from pathlib import Path
from collections import Counter

assert STUDENT_ID.strip(), "[Error] 請先填入 STUDENT_ID (你的學號)。"
assert SEED.strip(), "[Error] 請先填入 SEED (老師在課堂上公布)。"

# 從目前資料夾往上尋找含有 data/processed 的專案根目錄
# (notebook 放在 notebooks/ 或專案根目錄都可以)
start = Path(os.path.abspath(''))
project_root = next((p for p in [start, *start.parents] if (p / "data" / "processed").is_dir()), None)
assert project_root, f"[Error] 從 {start} 往上找不到 data/processed 資料夾。請確認本 notebook 放在你的專案資料夾內。"
PROCESSED_DATA_DIR = project_root / "data" / "processed"
INTERIM_DATA_DIR = project_root / "data" / "interim"
print(f"[Info] 專案根目錄: {project_root}")

def readable_md(json_path, position, chunk_text):
    """找出對應的 cleaned_*.md (data/interim/)，並計算這個 chunk 從第幾行開始"""
    md_name = json_path.name.replace("_processed.json", ".md")
    hits = sorted(INTERIM_DATA_DIR.rglob(md_name)) if INTERIM_DATA_DIR.is_dir() else []
    same_folder = [h for h in hits if h.parent.name == json_path.parent.name]   # 優先選同一個類別資料夾
    hits = same_folder or hits
    if not hits:
        return "(找不到，請改看 JSON)", None
    md_path = hits[0]
    md_text = md_path.read_text(encoding="utf-8")
    # 與 01_Data_Ingestion.ipynb 相同的切塊規則，逐塊累計行號
    line, k = 1, 0
    for part in re.split(r"\n(?=##\s)", md_text):
        if part.strip():
            if k == position and part.strip() == chunk_text.strip():
                return md_path.relative_to(project_root).as_posix(), line
            k += 1
        line += part.count("\n") + 1
    # 備用：直接搜尋第一行
    first = chunk_text.strip().splitlines()[0]
    for num, text_line in enumerate(md_text.splitlines(), start=1):
        if text_line.strip() == first.strip():
            return md_path.relative_to(project_root).as_posix(), num
    return md_path.relative_to(project_root).as_posix(), None

# 搜尋 data/processed 底下所有層級的子資料夾
json_files = sorted(PROCESSED_DATA_DIR.rglob("cleaned_*_processed.json"))
assert json_files, f"[Error] 在 {PROCESSED_DATA_DIR} 找不到 cleaned_*_processed.json，請先完成 01_Data_Ingestion.ipynb 的批次處理。"

def chunk_entropy(text):
    counts = Counter(text)
    return -sum((n / len(text)) * math.log2(n / len(text)) for n in counts.values())

# 讀取所有 cleaned chunks，並記下每個 chunk 在檔案中的位置 (第幾個)
chunks = []
for f in json_files:
    for position, record in enumerate(json.loads(f.read_text(encoding="utf-8"))):
        if record["text"].strip():
            md_file, md_line = readable_md(f, position, record["text"])
            chunks.append({"file": f.relative_to(PROCESSED_DATA_DIR).as_posix(), "md": md_file, "md_line": md_line, "position": position,
                           "text": record["text"], "length": len(record["text"]),
                           "entropy": chunk_entropy(record["text"])})

print(f"[Info] 讀取 {len(json_files)} 個 JSON 檔案，共 {len(chunks)} 個 chunks。")

In [ ]:
# 由「學號 + SEED」決定要分配哪一個 chunk (同樣的學號與 SEED，每次結果都一樣)
def pick(pool, label):
    # 1. 組成文字，例如 "M11412345|apple|long" (label: "long" = 很長的 chunk，"low" = 資訊熵很低的 chunk)
    key = f"{STUDENT_ID.strip().upper()}|{SEED.strip()}|{label}"
    # 2. 計算 SHA-256 雜湊值 (64 個十六進位字元)
    digest = hashlib.sha256(key.encode("utf-8")).hexdigest()
    # 3. 把雜湊值當成一個大數字，除以候選數量取餘數 → 決定選第幾個 chunk
    chosen = pool[int(digest, 16) % len(pool)]
    # 4. 驗證碼 = digest[:8]，也就是雜湊值的前 8 個字元
    verification_code = digest[:8]
    return chosen, verification_code

POOL_SIZE = 20
longest_pool = sorted(chunks, key=lambda c: c["length"], reverse=True)[:POOL_SIZE]
long_chunk, long_code = pick(longest_pool, "long")

# 資訊熵最低的候選，排除剛剛分到的長 chunk
low_pool = [c for c in sorted(chunks, key=lambda c: c["entropy"]) if c is not long_chunk][:POOL_SIZE]
low_chunk, low_code = pick(low_pool, "low")

def show(title, c, rank, code_):
    first_line = c["text"].strip().splitlines()[0]
    print(f"=== {title} ===")
    print(f"資料檔 (JSON): data/processed/{c['file']}")
    print(f"閱讀用檔案 (有正常換行): {c['md']}")
    if c["md_line"]:
        print(f"從第 {c['md_line']} 行開始 (VS Code 中按 Ctrl + G，輸入 {c['md_line']}，即可跳到這一行)")
    print(f"位置: 第 {c['position']} 個 chunk (從 0 開始數)，也就是 JSON 陣列中的第 {c['position'] + 1} 個物件")
    print(f"長度: {c['length']} 字元 | 資訊熵: {c['entropy']:.2f} | 候選排名: 第 {rank} 名 | 驗證碼: {code_}")
    print(f"搜尋用 (這個 chunk 的第一行): {first_line[:80]}")
    print("預覽 (前 400 字元):" if c["length"] > 400 else "內容:")
    print(c["text"][:400])
    print()

print(f"學號: {STUDENT_ID.strip().upper()}   SEED: {SEED.strip()}\n")
show("你的「很長」chunk", long_chunk, longest_pool.index(long_chunk) + 1, long_code)
show("你的「資訊熵很低」chunk", low_chunk, low_pool.index(low_chunk) + 1, low_code)
print("下一步：在 VS Code 打開上面的「閱讀用檔案」，按 Ctrl + G 輸入行號 (或按 Ctrl + F 搜尋「搜尋用」文字)，從該行讀到下一個 ## 標題為止，就是完整的 chunk。讀完後再填寫 DATA_REPORT.md 第 3.4 節 (含驗證碼與行號)。")